# Notebook 3: Group-by, risk and leverage by sector
*Kaggle Pandas lesson: "Grouping and Sorting"*

**Module question: what makes a stock risky?** Notebook 1 suggested that sector matters. Today we
measure it: volatility, leverage and beta *by sector*. The tool is `groupby`, the single most useful
idea in pandas: split the table into groups, compute something for each group, put the results back
together. Then we sort.

## Learning goals
* group rows with `groupby` and summarize each group with `count`, `size`, `mean`, `median`, `min`, `max`,
  several statistics at once with `agg`, or your own function with `apply`;
* group by two columns, recognize the **MultiIndex** this creates and flatten it with `reset_index`;
* sort a Series or DataFrame by value (`sort_values`) or by index (`sort_index`).

## Setup

In [ ]:
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/assacohen1/fin6040-pandas-data/main/"
firms = pd.read_csv(DATA_URL + "companies_2025.csv")

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 30)

Ratios from Notebook 2 (each notebook stands alone, so we recompute them):

In [ ]:
firms["leverage"] = firms.total_debt / (firms.total_debt + firms.market_cap)
firms["debt_to_assets"] = firms.total_debt / firms.total_assets
firms["gross_margin"] = (firms.sales - firms.cogs) / firms.sales

## In class

### 1. One group, one statistic
`groupby("sector")` splits the table into one piece per sector. Whatever you write after it is computed
for each piece. Counting the rows per group gives the same numbers as `value_counts` did:

In [ ]:
firms.groupby("sector").sector.count()

`size()` counts rows per group; `.column.count()` counts the **non-missing** values of that column. Same here, different when values are missing.

In [ ]:
firms.groupby("sector").size()

Now the statistic we actually care about: the average volatility in each sector.

In [ ]:
firms.groupby("sector").vol_2025.mean()

In [ ]:
firms.groupby("sector").vol_2025.median()

In [ ]:
firms.groupby("sector").market_cap.max()      # the largest firm in each sector, $ millions

A group whose values are all missing simply shows NaN. Financials have no cost of goods sold, so:

In [ ]:
firms.groupby("sector").gross_margin.median()

### 2. Your own function per group: `apply`
`apply` hands the function one group at a time, as a DataFrame. The file is sorted by market cap, so
the first row of each group is the largest firm in the sector. (`include_groups=False` keeps the
grouping column out of the pieces; newer pandas versions warn without it.)

In [ ]:
firms.groupby("sector").apply(lambda df: df.ticker.iloc[0], include_groups=False)

The most volatile stock in each sector, using `idxmax` from Notebook 2 inside each group:

In [ ]:
firms.groupby("sector").apply(lambda df: df.loc[df.vol_2025.idxmax(), "ticker"], include_groups=False)

### 3. Several statistics at once: `agg`
Pass a list of statistic names (as strings). The result is a DataFrame with one column per statistic.

In [ ]:
firms.groupby("sector").vol_2025.agg(["count", "mean", "median", "min", "max"])

### 4. Grouping by two columns: the MultiIndex
Pass a list of columns. The result has two levels of row labels, a **MultiIndex**. It looks nice but
is awkward to work with; `reset_index()` turns the levels back into ordinary columns.

In [ ]:
by_sec_exch = firms.groupby(["sector", "exchange"]).vol_2025.agg(["count", "mean"])
by_sec_exch

In [ ]:
type(by_sec_exch.index)

In [ ]:
by_sec_exch.reset_index()

### 5. Sorting
Group results come out in index order (alphabetical sectors). `sort_values` sorts by value, ascending
by default; `sort_index` goes back to index order. Sorting returns a new object; nothing changes in place.

In [ ]:
sector_vol = firms.groupby("sector").vol_2025.mean()
sector_vol.sort_values()

In [ ]:
sector_vol.sort_values(ascending=False)

In [ ]:
sector_vol.sort_index()

A DataFrame has many columns, so `sort_values` needs to know which one: `by=`. A list sorts by several columns, first the first.

In [ ]:
by_sec_exch.reset_index().sort_values(by=["sector", "mean"], ascending=False)

In [ ]:
firms.sort_values(by="vol_2025", ascending=False).head(10)[["ticker", "company", "sector", "market_cap", "vol_2025"]]

## Exercises

### Exercise 1: Firms per exchange

Create a Series `firms_per_exchange` whose index is the exchange and whose values are the number of firms, sorted by index (alphabetically).

<details><summary>Hint</summary>

`groupby("exchange").size()` then `sort_index()`.

</details>

In [ ]:
firms_per_exchange = ____

firms_per_exchange

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert firms_per_exchange.sum() == len(firms)
assert firms_per_exchange["NYSE"] == 1171
print("Looks right!")

### Exercise 2: Sector volatility

Create `sector_vol`: the mean `vol_2025` of each sector, most volatile sector first.

<details><summary>Hint</summary>

Group, take the mean of the column, then `sort_values(ascending=False)`.

</details>

In [ ]:
sector_vol = ____

sector_vol

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert len(sector_vol) == 11
assert sector_vol.index[0] == 'Health Care'
assert sector_vol.index[-1] == 'Utilities'
assert round(sector_vol["Utilities"], 4) == 0.2714
print("Looks right!")

### Exercise 3: Volatility range per sector

Create a DataFrame `vol_extremes`, indexed by sector, with two columns `min` and `max`: the lowest and highest `vol_2025` in the sector.

<details><summary>Hint</summary>

`agg` with a list of two names.

</details>

In [ ]:
vol_extremes = ____

vol_extremes

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert vol_extremes.shape == (11, 2)
assert list(vol_extremes.columns) == ["min", "max"]
print("Looks right!")

### Exercise 4: Sort by two columns

Create `sorted_extremes`: `vol_extremes` sorted in descending order by `min`, then by `max` as a tie-breaker. Which sector's *calmest* stock is still the most volatile?

<details><summary>Hint</summary>

`sort_values(by=[...], ascending=False)`.

</details>

In [ ]:
sorted_extremes = ____

sorted_extremes

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert sorted_extremes.index[0] == 'Energy'
assert sorted_extremes["min"].iloc[0] >= sorted_extremes["min"].iloc[-1]
print("Looks right!")

### Exercise 5: Sector leverage

Create `sector_lev`: the **median** `leverage` of each sector, most levered sector first. Compare its order with `sector_vol` from Exercise 2.

<details><summary>Hint</summary>

Same pattern as Exercise 2 with `median()` and the `leverage` column.

</details>

In [ ]:
sector_lev = ____

sector_lev

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert sector_lev.index[0] == 'Utilities'
assert round(sector_lev["Information Technology"], 4) == 0.0781
print("Looks right!")

### Exercise 6: Sector and exchange

Which sector-exchange combinations are the most common? Create `sector_exchange`, a Series with a MultiIndex (sector, exchange) holding the number of firms, largest first.

<details><summary>Hint</summary>

Group by a list of two columns, `size()`, then sort descending.

</details>

In [ ]:
sector_exchange = ____

sector_exchange

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert isinstance(sector_exchange.index, pd.MultiIndex)
assert sector_exchange.index[0] == ('Industrials', 'NYSE')
print("Looks right!")

### Exercise 7: The ten most volatile stocks

Create `top10_vol`: the 10 firms with the highest `vol_2025`, showing the columns `ticker`, `sector`, `market_cap`, `leverage`, `vol_2025`.

<details><summary>Hint</summary>

Sort the whole table descending, `head(10)`, then pick the columns with a list in brackets.

</details>

In [ ]:
top10_vol = ____

top10_vol

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert top10_vol.shape == (10, 5)
assert top10_vol.vol_2025.iloc[0] == firms.vol_2025.max()
print("Looks right!")

### Putting it together (worked example)
Three sector tables side by side. `pd.DataFrame` built from a dictionary of Series lines them up by
index label (the constructor from Notebook 1 again).

In [ ]:
dashboard = pd.DataFrame({
    "vol": firms.groupby("sector").vol_2025.mean(),
    "leverage": firms.groupby("sector").leverage.median(),
    "beta": firms.groupby("sector").beta_2025.mean(),
    "firms": firms.groupby("sector").size(),
}).sort_values(by="vol", ascending=False)
dashboard.round(3)

*Optional:* one line makes a chart of it.

In [ ]:
dashboard.vol.plot(kind="bar", title="Average volatility by sector, 2025");

## Finance insight

Utilities carries the highest financial leverage (median debt share
44%) yet ranks 11th of 11 in volatility (1 = most volatile); **Health Care**
and **Information Technology** are the most volatile sectors (mean volatility 63%
and 57%) with far less debt (Information Technology median debt share
8%). **Across sectors, debt does not explain risk; business
risk does.** Sectors with stable, predictable cash flows (regulated utilities, staples, real estate) can
afford a lot of debt and choose to use it; sectors whose cash flows are uncertain stay lightly levered.
Capital structure is a *response* to business risk (this is the trade-off theory of capital structure).

Beta ranks the sectors in a similar order but with a much narrower spread than volatility: a large part
of a sector's volatility is idiosyncratic risk that a diversified portfolio removes and that beta,
by construction, ignores.